In [7]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma

# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


# Chat model
model = ChatOllama(model="qwen3")


# Load the PDF document
file_path = r"doc\\llama2-research-paper.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()
print("Total pages:", len(pages))

# Split the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))


"""
        Chroma is a vector database, but in LangChain it is exposed through the Chroma() 
        vector store interface. Therefore, we usually refer to it as a vector store while 
        writing RAG applications.
"""

# https://www.trychroma.com/ ----> This cloud base chroma DB
# Create a Chroma vector store
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

# Add the document chunks to the vector store
document_ids = vector_store.add_documents(chunks)
print("Documents added:", len(document_ids))
print("Total documents stored:", vector_store._collection.count())

#  To ceate retriever, we can use the vector store's as_retriever() method.
#  This method allows us to specify the search type and any additional search parameters. 
#  In this case, we are using similarity search with a parameter k=5, which means we want 
# to retrieve the top 5 most similar documents for a given query.
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)


# --------------------------------------------------
# 8. Test retriever
# --------------------------------------------------
#query = "What is the architecture of Llama 2?"
#retrieved_documents = retriever.invoke(query)
#for i, document in enumerate(retrieved_documents, start=1):
#    print(f"--------------retrive_document_{i}----------------------")
#    print(document.page_content[:100])
#    print(document.metadata)

prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

# --------------------------------------------------
# 10. Format documents
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )

# --------------------------------------------------
# 12. RAG chain
# --------------------------------------------------

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

# --------------------------------------------------
# 13. Ask question
# --------------------------------------------------

answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3135.28it/s]


Total pages: 77
Total chunks: 175
Documents added: 175
Total documents stored: 350

Final answer:

The architecture of Llama 2 is described as an **autoregressive language model** that uses an **optimized transformer architecture**. The tuned versions of the model incorporate **supervised fine-tuning (SFT)** and **reinforcement learning with human feedback (RLHF)** to align with human preferences for helpfulness and safety.
